# Time series benchmarking tutorial (Hydrology CAMELS US with Chronos 2)

This is the **main DS5110 final notebook**.

## Stage 1 (this update): Data input and preprocessing

- Input file: `data/raw/BasicInputTimeSeries_us.csv`
- Task: single-step forecasting setup preparation
- Input mode: strictly univariate (`QObs(mm/d)` only)
- Lookback window: 30
- Split: 80% train / 10% val / 10% test (chronological per basin)

In this stage, we focus only on loading, validating, cleaning, splitting, and saving processed artifacts for later Chronos-2 evaluation.


In [5]:
!git clone https://github.com/JunyangHe/timeseries_benchmark_tutorial.git

Cloning into 'timeseries_benchmark_tutorial'...
remote: Enumerating objects: 44, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 44 (delta 0), reused 0 (delta 0), pack-reused 43 (from 3)
Receiving objects: 100% (44/44), 89.03 MiB | 15.17 MiB/s, done.
Resolving deltas: 100% (6/6), done.


In [6]:
%cd timeseries_benchmark_tutorial
!pip install -q -r requirements.txt

/content/timeseries_benchmark_tutorial


In [7]:
!unzip "./data/raw/BasicInputTimeSeries_us.zip"

Archive:  ./data/raw/BasicInputTimeSeries_us.zip
  inflating: BasicInputTimeSeries_us.csv  


In [8]:
# Install dependencies (Colab-friendly, configurable)
import os
import subprocess
import sys
from pathlib import Path


def find_in_cwd_or_parents(filename: str, start: Path, max_levels: int = 6):
    cur = start.resolve()
    for _ in range(max_levels + 1):
        candidate = cur / filename
        if candidate.exists():
            return candidate
        if cur.parent == cur:
            break
        cur = cur.parent
    return None


req_override = os.getenv('REQUIREMENTS_PATH', '').strip()
req_path = Path(req_override).expanduser() if req_override else find_in_cwd_or_parents('requirements.txt', Path.cwd())

if req_path is not None and req_path.exists():
    print(f'Installing from: {req_path.resolve()}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req_path)])
else:
    print('requirements.txt not found. Installing fallback packages...')
    subprocess.check_call([
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        'pandas>=2.0',
        'numpy>=1.24',
        'matplotlib>=3.7',
        'pyarrow>=14.0',
        'chronos-forecasting>=2.0',
        'torch>=2.1',
        'tqdm>=4.66',
        'pyyaml>=6.0',
    ])

print('Dependency installation complete.')

Installing from: /content/timeseries_benchmark_tutorial/requirements.txt
Dependency installation complete.


In [2]:
import json
import os
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

# Optional Colab Drive mount:
# from google.colab import drive
# drive.mount('/content/drive')


def find_project_root(start: Path, max_levels: int = 8) -> Path:
    """Pick nearest parent that looks like this repo."""
    cur = start.resolve()
    for _ in range(max_levels + 1):
        if (cur / 'src').exists() and (cur / 'notebooks').exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    return start.resolve()


def resolve_input_csv(project_root: Path, relative_default: Path) -> Path:
    """
    Resolve input CSV by priority:
    1) INPUT_CSV_PATH env var
    2) <project_root>/<relative_default>
    3) <cwd>/<relative_default>
    4) first recursive match by filename under project_root, then cwd
    """
    env_path = os.getenv('INPUT_CSV_PATH', '').strip()
    if env_path:
        return Path(env_path).expanduser()

    candidate = project_root / relative_default
    if candidate.exists():
        return candidate

    candidate = Path.cwd() / relative_default
    if candidate.exists():
        return candidate

    filename = relative_default.name
    match = next(project_root.rglob(filename), None)
    if match is not None:
        return match

    match = next(Path.cwd().rglob(filename), None)
    if match is not None:
        return match

    return project_root / relative_default


# ----------------------------
# User-editable runtime config
# ----------------------------
DATA_RELATIVE_PATH = Path('./data/raw/BasicInputTimeSeries_us.csv')
MODEL_ID = os.getenv('CHRONOS_MODEL_ID', 'amazon/chronos-2')
LOOKBACK_WINDOW = 30
FORECAST_HORIZON = 1
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1

TIMESTAMP_COL = 'Year_Mnth_Day'
ID_COL = 'basin_id'
TARGET_COL = 'QObs(mm/d)'
REQUIRED_COLUMNS = [TIMESTAMP_COL, ID_COL, TARGET_COL]

PROJECT_ROOT = find_project_root(Path.cwd())
INPUT_CSV_PATH = resolve_input_csv(PROJECT_ROOT, DATA_RELATIVE_PATH)
OUTPUT_ROOT = PROJECT_ROOT

OUTPUT_DIRS = [
    OUTPUT_ROOT / 'metadata',
    OUTPUT_ROOT / 'results',
    OUTPUT_ROOT / 'results' / 'figures',
    OUTPUT_ROOT / 'data' / 'processed',
]
for d in OUTPUT_DIRS:
    d.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Resolved input path:', INPUT_CSV_PATH)
print('Output root:', OUTPUT_ROOT.resolve())
print('Model ID:', MODEL_ID)

Project root: /content
Resolved input path: /content/data/raw/BasicInputTimeSeries_us.csv
Output root: /content
Model ID: amazon/chronos-2


In [9]:
import sys

# Make src/ imports work in both local and Colab contexts without fixed absolute paths.
src_candidates = [
    PROJECT_ROOT / 'src',
    Path.cwd() / 'src',
    Path.cwd().parent / 'src',
]

src_path = next((p for p in src_candidates if p.exists()), None)
if src_path is None:
    raise FileNotFoundError('Could not find src/ directory. Ensure notebook runs inside the repository.')

sys.path.insert(0, str(src_path.resolve()))
print('Using src path:', src_path.resolve())

from chronos import Chronos2Pipeline
from data_utils import (
    DataSpec,
    build_context_and_test,
    chronological_split_by_id,
    load_and_prepare_csv,
    to_chronos_df,
)
from chronos_eval import rolling_one_step_predictions
from metrics import rmse
from plotting import plot_actual_vs_predicted

Using src path: /content/timeseries_benchmark_tutorial/src


In [10]:
# Load quick preview and validate required columns
if not INPUT_CSV_PATH.exists():
    raise FileNotFoundError(
        'Input CSV not found. Set env var INPUT_CSV_PATH or place file at data/raw/BasicInputTimeSeries_us.csv'
    )

preview_df = pd.read_csv(INPUT_CSV_PATH, nrows=5)
print('Preview columns:', list(preview_df.columns))
print(preview_df.head())

missing_required = [c for c in REQUIRED_COLUMNS if c not in preview_df.columns]
if missing_required:
    raise ValueError(f'Missing required columns: {missing_required}')

print('Required columns found:', REQUIRED_COLUMNS)

FileNotFoundError: Input CSV not found. Set env var INPUT_CSV_PATH or place file at data/raw/BasicInputTimeSeries_us.csv

In [ ]:
# Load full dataset and preprocess
spec = DataSpec(
    timestamp_col=TIMESTAMP_COL,
    id_col=ID_COL,
    target_col=TARGET_COL,
    drop_unnamed_first_col=True,  # user requirement
)

raw_df = load_and_prepare_csv(INPUT_CSV_PATH, spec)

print('Prepared rows:', len(raw_df))
print('Prepared basins:', raw_df[ID_COL].nunique())
print('Date range:', raw_df[TIMESTAMP_COL].min(), 'to', raw_df[TIMESTAMP_COL].max())
print(raw_df[[TIMESTAMP_COL, ID_COL, TARGET_COL]].head())

In [ ]:
# Split each basin chronologically: 80/10/10
splits = chronological_split_by_id(
    raw_df,
    id_col=ID_COL,
    timestamp_col=TIMESTAMP_COL,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    test_ratio=TEST_RATIO,
)

train_df_raw = splits['train']
val_df_raw = splits['val']
test_df_raw = splits['test']

print('Train rows:', len(train_df_raw))
print('Val rows:', len(val_df_raw))
print('Test rows:', len(test_df_raw))

# Convert to Chronos long format (still only data prep stage)
train_df = to_chronos_df(train_df_raw, id_col=ID_COL, timestamp_col=TIMESTAMP_COL, target_col=TARGET_COL)
val_df = to_chronos_df(val_df_raw, id_col=ID_COL, timestamp_col=TIMESTAMP_COL, target_col=TARGET_COL)
test_df = to_chronos_df(test_df_raw, id_col=ID_COL, timestamp_col=TIMESTAMP_COL, target_col=TARGET_COL)

print('\nChronos-formatted sample:')
print(train_df.head())

In [ ]:
# Save processed artifacts for reproducible next stages
processed_dir = OUTPUT_ROOT / 'data' / 'processed'

train_df.to_parquet(processed_dir / 'train_chronos.parquet', index=False)
val_df.to_parquet(processed_dir / 'val_chronos.parquet', index=False)
test_df.to_parquet(processed_dir / 'test_chronos.parquet', index=False)

# Small CSV heads for quick inspection
train_df.head(1000).to_csv(processed_dir / 'train_preview_1000.csv', index=False)
val_df.head(1000).to_csv(processed_dir / 'val_preview_1000.csv', index=False)
test_df.head(1000).to_csv(processed_dir / 'test_preview_1000.csv', index=False)

print('Saved processed splits to:', processed_dir.resolve())

In [ ]:
# Quick data sanity visualization (preprocessing stage)
# Plot target over time for one example basin.
example_id = str(train_df['id'].iloc[0])
example_series = train_df[train_df['id'] == example_id].sort_values('timestamp').tail(365)

plt.figure(figsize=(12, 4))
plt.plot(example_series['timestamp'], example_series['target'])
plt.title(f'Example basin {example_id}: last 365 train points of {TARGET_COL}')
plt.xlabel('Date')
plt.ylabel(TARGET_COL)
plt.tight_layout()

prep_fig_path = OUTPUT_ROOT / 'results' / 'figures' / 'preprocess_example_series.png'
plt.savefig(prep_fig_path, dpi=150)
plt.show()
print('Saved:', prep_fig_path)

In [ ]:
# Save preprocessing summary for reproducibility
preprocess_summary = {
    'project_root': str(PROJECT_ROOT),
    'input_csv_path': str(INPUT_CSV_PATH),
    'timestamp_column': TIMESTAMP_COL,
    'id_column': ID_COL,
    'target_column': TARGET_COL,
    'drop_unnamed_first_column': True,
    'fill_strategy': 'forward_fill_within_basin',
    'train_ratio': TRAIN_RATIO,
    'val_ratio': VAL_RATIO,
    'test_ratio': TEST_RATIO,
    'lookback_window': LOOKBACK_WINDOW,
    'forecast_horizon': FORECAST_HORIZON,
    'prepared_rows': int(len(raw_df)),
    'prepared_basins': int(raw_df[ID_COL].nunique()),
    'train_rows': int(len(train_df)),
    'val_rows': int(len(val_df)),
    'test_rows': int(len(test_df)),
}

summary_path = OUTPUT_ROOT / 'results' / 'preprocessing_summary.json'
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(preprocess_summary, f, indent=2)

print('Saved:', summary_path)
print(json.dumps(preprocess_summary, indent=2))

In [ ]:
# Stage 2: Chronos-2 inference, RMSE, and Actual vs Predicted
import torch

device_map = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device_map)

context_df, eval_test_df = build_context_and_test(train_df, val_df, test_df)
print('Context rows:', len(context_df), '| Test rows:', len(eval_test_df))

pipeline = Chronos2Pipeline.from_pretrained(MODEL_ID, device_map=device_map)

pred_test_df = rolling_one_step_predictions(
    pipeline=pipeline,
    context_df=context_df,
    test_df=eval_test_df,
    lookback_window=LOOKBACK_WINDOW,
    quantile_levels=[0.5],
)

if pred_test_df.empty:
    raise RuntimeError('Prediction output is empty. Check the split and input data.')

overall_test_rmse = rmse(pred_test_df['actual'].values, pred_test_df['predicted'].values)
print('Overall test RMSE:', overall_test_rmse)

pred_path = OUTPUT_ROOT / 'results' / 'predictions_test.csv'
pred_test_df.to_csv(pred_path, index=False)
print('Saved:', pred_path)

fig_path = OUTPUT_ROOT / 'results' / 'figures' / 'actual_vs_predicted.png'
plot_actual_vs_predicted(pred_test_df, str(fig_path))
print('Saved:', fig_path)

# Display Actual vs Predicted directly in notebook output
plt.figure(figsize=(7, 7))
plt.scatter(pred_test_df['actual'], pred_test_df['predicted'], alpha=0.25, s=8)
line_min = min(pred_test_df['actual'].min(), pred_test_df['predicted'].min())
line_max = max(pred_test_df['actual'].max(), pred_test_df['predicted'].max())
plt.plot([line_min, line_max], [line_min, line_max], linestyle='--')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Actual vs Predicted (Test Set)')
plt.tight_layout()
plt.show()

benchmark_result = {
    'dataset_name': 'CAMELS-US',
    'model_name': 'chronos-2',
    'target_variable': TARGET_COL,
    'lookback_window': LOOKBACK_WINDOW,
    'forecast_horizon': FORECAST_HORIZON,
    'model_parameters': {
        'model_id': MODEL_ID,
        'prediction_length': 1,
        'quantile_levels': [0.5],
        'device_map': device_map,
    },
    'rmse': float(overall_test_rmse),
    'notes': 'Overall RMSE on chronological 10% test split across all basins.',
}

benchmark_path = OUTPUT_ROOT / 'results' / 'benchmark_results.json'
with open(benchmark_path, 'w', encoding='utf-8') as f:
    json.dump(benchmark_result, f, indent=2)

print('Saved:', benchmark_path)
print(json.dumps(benchmark_result, indent=2))

In [ ]:
# Update metadata files with observed dataset counts
# This keeps machine-readable metadata consistent with the real preprocessed input.
dataset_card_path = OUTPUT_ROOT / 'metadata' / 'dataset_card.json'
ts_chars_path = OUTPUT_ROOT / 'metadata' / 'ts_characteristics.json'
experiment_cfg_path = OUTPUT_ROOT / 'metadata' / 'experiment_config.json'

if dataset_card_path.exists():
    with open(dataset_card_path, 'r', encoding='utf-8') as f:
        dataset_card = json.load(f)
    dataset_card['number_of_locations'] = int(raw_df[ID_COL].nunique())
    dataset_card['number_of_time_points'] = int(len(raw_df))
    with open(dataset_card_path, 'w', encoding='utf-8') as f:
        json.dump(dataset_card, f, indent=2)

if ts_chars_path.exists():
    with open(ts_chars_path, 'r', encoding='utf-8') as f:
        ts_chars = json.load(f)
    ts_chars['number_of_instances'] = int(raw_df[ID_COL].nunique())
    with open(ts_chars_path, 'w', encoding='utf-8') as f:
        json.dump(ts_chars, f, indent=2)

if experiment_cfg_path.exists():
    with open(experiment_cfg_path, 'r', encoding='utf-8') as f:
        exp_cfg = json.load(f)
    exp_cfg['lookback_window'] = int(LOOKBACK_WINDOW)
    exp_cfg['forecast_horizon'] = int(FORECAST_HORIZON)
    exp_cfg['evaluation_metrics'] = ['rmse']
    with open(experiment_cfg_path, 'w', encoding='utf-8') as f:
        json.dump(exp_cfg, f, indent=2)

print('Metadata refresh complete.')

## Outputs after full run

- `data/processed/train_chronos.parquet`
- `data/processed/val_chronos.parquet`
- `data/processed/test_chronos.parquet`
- `data/processed/*_preview_1000.csv`
- `results/preprocessing_summary.json`
- `results/predictions_test.csv`
- `results/benchmark_results.json`
- `results/figures/preprocess_example_series.png`
- `results/figures/actual_vs_predicted.png`
- refreshed metadata files in `metadata/`

## Optional Colab overrides

Use environment variables if your files are in custom locations:

```python
import os
os.environ['INPUT_CSV_PATH'] = '/content/BasicInputTimeSeries_us.csv'
os.environ['REQUIREMENTS_PATH'] = '/content/requirements.txt'
os.environ['CHRONOS_MODEL_ID'] = 'amazon/chronos-2'
```